<a href="https://colab.research.google.com/github/dan-the-man7/lab_4_/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%pip install -q openai python-dotenv pandas matplotlib

import os, json, time, re, random

API_KEY = None
try:                                   # --- Google Colab (Secrets panel) ---
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except Exception:                      # --- Local (.env file) ---
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.environ.get("Groq_API_key")

assert API_KEY, (
    "No API key found. In Colab add a secret named GROQ_API_KEY (key icon, "
    "left sidebar) and switch on 'Notebook access'. Locally, create a .env file."
)
print("Key loaded:", API_KEY[:4] + "..." + API_KEY[-4:])   # never print the whole

# OpenAI-compatible client pointed at Groq
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready. Model:", MODEL)

Key loaded: gsk_...ZNuR
Client ready. Model: llama-3.3-70b-versatile


In [6]:
TOKEN_LOG = []          # (label, prompt_tokens, completion_tokens, total_tokens)


def ask_llm(user_prompt,
            system_prompt="You are a helpful assistant.",
            temperature=0.7,
            max_tokens=500,
            label="",
            retries=6):
    """Send one single-turn chat request and return the assistant's text."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            u = response.usage
            TOKEN_LOG.append((label or "unlabelled",
                              u.prompt_tokens, u.completion_tokens, u.total_tokens))
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt + random.random()
                print(f"   [rate limited — sleeping {wait:.1f}s]")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"ask_llm failed after {retries} attempts")


# A first call, looking at the RAW response object so we can see the
# anatomy: roles, the choices list, and the usage accounting.

raw = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You explain finance simply, for a non-expert."},
        {"role": "user",   "content": "In two sentences, what is microfinance?"},
    ],
    temperature=0.7,
    max_tokens=150,
)

print("Answer:\n", raw.choices[0].message.content.strip())
print("\nRole of the reply:", raw.choices[0].message.role)
print("Finish reason     :", raw.choices[0].finish_reason)
print("\nUsage:")
print("  prompt_tokens     =", raw.usage.prompt_tokens)
print("  completion_tokens =", raw.usage.completion_tokens)
print("  total_tokens      =", raw.usage.total_tokens)

print("\nQuestion passed through ask_llm():\n",
      ask_llm("Name one risk a microfinance lender faces.",
              label="smoke_test", max_tokens=60))

Answer:
 Microfinance is a type of financial service that provides small loans, savings, and other financial products to individuals or businesses that do not have access to traditional banking services, often in developing countries or low-income communities. The goal of microfinance is to help people with limited financial resources to start or grow their own businesses, improve their economic well-being, and gain financial stability, often with loans as small as a few hundred dollars.

Role of the reply: assistant
Finish reason     : stop

Usage:
  prompt_tokens     = 55
  completion_tokens = 86
  total_tokens      = 141

Question passed through ask_llm():
 One risk a microfinance lender faces is default risk, which is the risk that borrowers will be unable to repay their loans, resulting in financial losses for the lender.


In [7]:
QUESTION = "Suggest a name for a savings product for market traders in Accra."
N_RUNS = 5

runs = {}
for temp in (0.0, 1.2):
  print(f"Running {N_RUNS} calls at temperature={temp} ...")
  runs[temp] = [
      ask_llm(QUESTION, temperature=temp, max_tokens=40, label=f"temp_{temp}")
      for _ in range(N_RUNS)
  ]

for temp, answers in runs.items():
  print("\n")
  print(f"Temperature = {temp}")
  print("\n")
  for i, a in enumerate(answers, 1):
    print(f"[{i}] {a}")

print(f"\n Distinct answers at temperature 0.0 : {len(set(runs[0.0]))} / {N_RUNS}")
print(f"\n Distinct answers at temperature 1.2 : {len(set(runs[1.2]))} / {N_RUNS}")

Running 5 calls at temperature=0.0 ...
Running 5 calls at temperature=1.2 ...


Temperature = 0.0


[1] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this
[2] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this
[3] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this
[4] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this
[5] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this


Temperature = 1.2


[1] For a savings product targeting market traders in Accra, I would suggest the following name options:

1. **Ma